In [1]:
import shutil, os

os.makedirs("provenance", exist_ok=True)
shutil.copy("/opt/ml/metadata/resource-metadata.json", "provenance/sagemaker-resource-metadata.json")

# Verifica que se copió bien y revisa el ResourceArn
import json
with open("provenance/sagemaker-resource-metadata.json") as f:
    meta = json.load(f)
print(meta.get("ResourceArn"))

arn:aws:sagemaker:us-east-1:214844251412:notebook-instance/analisis-de-sentimientos


## Conectarse al mlflow

In [2]:
!pip install mlflow -q

In [3]:
import mlflow

mlflow.set_tracking_uri("http://ec2-100-26-91-142.compute-1.amazonaws.com:5000")
mlflow.set_experiment("nlp-lab2-sentiment140")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1789655162210, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789655162210, lifecycle_stage='active', name='nlp-lab2-sentiment140', tags={}, trace_location=None, workspace='default'>

## P_ELONGATION

In [7]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "adilbekovich/Sentiment140Twitter",
    revision="b6037e127257d95b9b23d31f78b264b9ebe697fd"
)
train = dataset["train"]

train_df = train.to_pandas()
train_df["index"] = train_df.index

print(train_df.shape)

Generating train split: 1360000 examples [00:02, 642385.79 examples/s]
Generating test split: 240000 examples [00:00, 658912.38 examples/s]


(1360000, 3)


In [8]:
import mlflow

mlflow.set_tracking_uri("http://ec2-100-26-91-142.compute-1.amazonaws.com:5000")
mlflow.set_experiment("nlp-lab2-sentiment140")

protocol_run_id = "5159d4c11e934382ad32d19269f79112"

partitions_path = mlflow.artifacts.download_artifacts(run_id=protocol_run_id, artifact_path="protocol/partitions.csv")
partitions = pd.read_csv(partitions_path)
print(partitions.shape)

(200000, 2)


In [9]:
import json
with open("provenance/sagemaker-resource-metadata.json") as f:
    meta = json.load(f)
print(meta.get("ResourceArn"))

arn:aws:sagemaker:us-east-1:214844251412:notebook-instance/analisis-de-sentimientos


In [10]:
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
import sklearn

NEGATORS = ["not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nor"]

def preprocess_text(text, stopwords_mode="keep", lemmatize=False,
                     elongation="keep", emoji_mode="keep"):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "url", text)
    text = re.sub(r"@\w+", "user", text)

    if elongation == "normalize":
        text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    if emoji_mode == "text":
        import emoji as emoji_lib
        text = emoji_lib.demojize(text, delimiters=(" ", " "))

    text = re.sub(r"\s+", " ", text).strip()

    if stopwords_mode == "keep" and not lemmatize:
        return text

    doc = nlp(text)
    tokens = []
    for token in doc:
        word = token.text
        is_stop = token.is_stop or word in NEGATORS
        if stopwords_mode == "remove" and is_stop:
            continue
        if stopwords_mode == "remove_preserve_negation" and is_stop and word not in NEGATORS:
            continue
        if lemmatize:
            word = token.lemma_
        tokens.append(word)
    return " ".join(tokens)

def make_build_fn(preprocess_kwargs):
    def build_fn(X_train, y_train):
        X_train_proc = [preprocess_text(t, **preprocess_kwargs) for t in X_train]
        pipeline = Pipeline([
            ("vectorizer", CountVectorizer(ngram_range=(1, 1))),
            ("clf", LogisticRegression())
        ])
        pipeline.fit(X_train_proc, y_train)

        class Wrapped:
            def predict(self, X):
                X_proc = [preprocess_text(t, **preprocess_kwargs) for t in X]
                return pipeline.predict(X_proc)
        return Wrapped()
    return build_fn

def evaluate_pipeline(build_pipeline_fn, train_df, partitions_df):
    base = train_df[["index", "text", "label"]]
    merged = base.merge(partitions_df, on="index", how="inner")
    assert len(merged) == len(partitions_df), "La muestra no coincide con partitions.csv"

    fold_scores = []
    for fold_k in sorted(merged["fold"].unique()):
        train_part = merged[merged["fold"] != fold_k]
        val_part = merged[merged["fold"] == fold_k]
        X_train, y_train = train_part["text"].tolist(), train_part["label"].values
        X_val, y_val = val_part["text"].tolist(), val_part["label"].values
        model = build_pipeline_fn(X_train, y_train)
        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred, average="macro")
        fold_scores.append(f1)
        print(f"  Fold {fold_k}: Macro-F1 = {f1:.4f}")

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores, ddof=0)
    return fold_scores, mean_f1, std_f1

def base_config():
    return {
        "preprocessing": {
            "lowercase": True, "url": "token:url", "mention": "token:user",
            "whitespace": "normalize", "stopwords": "keep", "negators": [],
            "lemmatize": False, "elongation": "keep", "elongation_spec": None,
            "emoji": "keep", "emoji_spec": None, "resources": {}, "additional": {}
        },
        "representation": {
            "type": "bow", "ngram_range": [1, 1], "library": "sklearn",
            "library_version": sklearn.__version__, "spacy_model": None,
            "spacy_model_version": None, "parameters": {}
        },
        "classifier": {
            "type": "logistic_regression", "library": "sklearn",
            "library_version": sklearn.__version__, "parameters": {}
        }
    }

def log_experiment_run(run_name, lab_experiment_id, lab_stage, configuration_id,
                        config_json, scores, mean_f1, std_f1, member_id="E01"):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("lab_run_type", "experiment")
        mlflow.set_tag("lab_protocol_run_id", protocol_run_id)
        mlflow.set_tag("lab_experiment_id", lab_experiment_id)
        mlflow.set_tag("lab_stage", lab_stage)
        mlflow.set_tag("lab_member_id", member_id)
        mlflow.set_tag("lab_configuration_id", configuration_id)
        mlflow.set_tag("notebook_arn", meta.get("ResourceArn"))

        for i, f1 in enumerate(scores):
            mlflow.log_metric(f"macro_f1_fold_{i}", f1)
        mlflow.log_metric("macro_f1_mean", mean_f1)
        mlflow.log_metric("macro_f1_std", std_f1)

        with open("configuration.json", "w") as f:
            json.dump(config_json, f, indent=2)
        mlflow.log_artifact("configuration.json", artifact_path="run")
        mlflow.log_artifact("provenance/sagemaker-resource-metadata.json", artifact_path="provenance")

        print(f"{lab_experiment_id} run ID:", run.info.run_id)
        return run.info.run_id

In [13]:
# ===== SETUP COMPLETO =====
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
import json
import sklearn
import mlflow
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

MEMBER_ID = "E02"

# dataset completo
dataset = load_dataset("adilbekovich/Sentiment140Twitter",
                        revision="b6037e127257d95b9b23d31f78b264b9ebe697fd")
train_df = dataset["train"].to_pandas()
train_df["index"] = train_df.index
print("train_df:", train_df.shape)

# mlflow + protocolo existente
mlflow.set_tracking_uri("http://ec2-100-26-91-142.compute-1.amazonaws.com:5000")
mlflow.set_experiment("nlp-lab2-sentiment140")
protocol_run_id = "5159d4c11e934382ad32d19269f79112"

partitions_path = mlflow.artifacts.download_artifacts(run_id=protocol_run_id, artifact_path="protocol/partitions.csv")
partitions = pd.read_csv(partitions_path)
print("partitions:", partitions.shape)

# tu ARN
with open("provenance/sagemaker-resource-metadata.json") as f:
    meta = json.load(f)
print("ResourceArn:", meta.get("ResourceArn"))

NEGATORS = ["not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nor"]

def preprocess_text(text, stopwords_mode="keep", lemmatize=False, elongation="keep", emoji_mode="keep"):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "url", text)
    text = re.sub(r"@\w+", "user", text)
    if elongation == "normalize":
        text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    if emoji_mode == "text":
        import emoji as emoji_lib
        text = emoji_lib.demojize(text, delimiters=(" ", " "))
    text = re.sub(r"\s+", " ", text).strip()
    if stopwords_mode == "keep" and not lemmatize:
        return text
    doc = nlp(text)
    tokens = []
    for token in doc:
        word = token.text
        is_stop = token.is_stop or word in NEGATORS
        if stopwords_mode == "remove" and is_stop:
            continue
        if stopwords_mode == "remove_preserve_negation" and is_stop and word not in NEGATORS:
            continue
        if lemmatize:
            word = token.lemma_
        tokens.append(word)
    return " ".join(tokens)

def make_build_fn(preprocess_kwargs):
    def build_fn(X_train, y_train):
        X_train_proc = [preprocess_text(t, **preprocess_kwargs) for t in X_train]
        pipeline = Pipeline([("vectorizer", CountVectorizer(ngram_range=(1, 1))), ("clf", LogisticRegression())])
        pipeline.fit(X_train_proc, y_train)
        class Wrapped:
            def predict(self, X):
                X_proc = [preprocess_text(t, **preprocess_kwargs) for t in X]
                return pipeline.predict(X_proc)
        return Wrapped()
    return build_fn

def evaluate_pipeline(build_pipeline_fn, train_df, partitions_df):
    base = train_df[["index", "text", "label"]]
    merged = base.merge(partitions_df, on="index", how="inner")
    assert len(merged) == len(partitions_df)
    fold_scores = []
    for fold_k in sorted(merged["fold"].unique()):
        train_part = merged[merged["fold"] != fold_k]
        val_part = merged[merged["fold"] == fold_k]
        model = build_pipeline_fn(train_part["text"].tolist(), train_part["label"].values)
        y_pred = model.predict(val_part["text"].tolist())
        f1 = f1_score(val_part["label"].values, y_pred, average="macro")
        fold_scores.append(f1)
        print(f"  Fold {fold_k}: Macro-F1 = {f1:.4f}")
    return fold_scores, np.mean(fold_scores), np.std(fold_scores, ddof=0)

def base_config():
    return {
        "preprocessing": {"lowercase": True, "url": "token:url", "mention": "token:user",
                           "whitespace": "normalize", "stopwords": "keep", "negators": [],
                           "lemmatize": False, "elongation": "keep", "elongation_spec": None,
                           "emoji": "keep", "emoji_spec": None, "resources": {}, "additional": {}},
        "representation": {"type": "bow", "ngram_range": [1, 1], "library": "sklearn",
                            "library_version": sklearn.__version__, "spacy_model": None,
                            "spacy_model_version": None, "parameters": {}},
        "classifier": {"type": "logistic_regression", "library": "sklearn",
                       "library_version": sklearn.__version__, "parameters": {}}
    }

def log_experiment_run(run_name, lab_experiment_id, lab_stage, configuration_id,
                        config_json, scores, mean_f1, std_f1, member_id=MEMBER_ID):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("lab_run_type", "experiment")
        mlflow.set_tag("lab_protocol_run_id", protocol_run_id)
        mlflow.set_tag("lab_experiment_id", lab_experiment_id)
        mlflow.set_tag("lab_stage", lab_stage)
        mlflow.set_tag("lab_member_id", member_id)
        mlflow.set_tag("lab_configuration_id", configuration_id)
        mlflow.set_tag("notebook_arn", meta.get("ResourceArn"))
        for i, f1 in enumerate(scores):
            mlflow.log_metric(f"macro_f1_fold_{i}", f1)
        mlflow.log_metric("macro_f1_mean", mean_f1)
        mlflow.log_metric("macro_f1_std", std_f1)
        with open("configuration.json", "w") as f:
            json.dump(config_json, f, indent=2)
        mlflow.log_artifact("configuration.json", artifact_path="run")
        mlflow.log_artifact("provenance/sagemaker-resource-metadata.json", artifact_path="provenance")
        print(f"{lab_experiment_id} run ID:", run.info.run_id)
        return run.info.run_id

print("SETUP COMPLETO")

train_df: (1360000, 3)


partitions: (200000, 2)
ResourceArn: arn:aws:sagemaker:us-east-1:214844251412:notebook-instance/analisis-de-sentimientos
SETUP COMPLETO


In [14]:
config = base_config()
config["preprocessing"]["elongation"] = "normalize"
config["preprocessing"]["elongation_spec"] = "reduce_repeated_chars_to_2"

build_fn = make_build_fn({"elongation": "normalize"})
print("Evaluando P_ELONGATION...")
scores, mean_f1, std_f1 = evaluate_pipeline(build_fn, train_df, partitions)
print(f"P_ELONGATION -> mean={mean_f1:.4f}, std={std_f1:.4f}")

log_experiment_run("P_ELONGATION", "P_ELONGATION", "preprocessing", "CFG_P_ELONGATION",
                    config, scores, mean_f1, std_f1)

Evaluando P_ELONGATION...


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 0: Macro-F1 = 0.7833


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 1: Macro-F1 = 0.7844


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 2: Macro-F1 = 0.7828
P_ELONGATION -> mean=0.7835, std=0.0007
P_ELONGATION run ID: b93630d04caf4808aaedcf1eaff00c21
🏃 View run P_ELONGATION at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1/runs/b93630d04caf4808aaedcf1eaff00c21
🧪 View experiment at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1


'b93630d04caf4808aaedcf1eaff00c21'

## P_LEMMA

In [15]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q

import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

NEGATORS = ["not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nor"]

print("spaCy listo:", nlp.pipe_names)

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
spaCy listo: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer']


In [16]:
config = base_config()
config["preprocessing"]["lemmatize"] = True

build_fn = make_build_fn({"lemmatize": True})

print("Evaluando P_LEMMA... (esta es la más lenta, corre spaCy sobre ~133k textos por fold, puede tardar varios minutos)")
scores, mean_f1, std_f1 = evaluate_pipeline(build_fn, train_df, partitions)
print(f"P_LEMMA -> mean={mean_f1:.4f}, std={std_f1:.4f}")

log_experiment_run("P_LEMMA", "P_LEMMA", "preprocessing", "CFG_P_LEMMA",
                    config, scores, mean_f1, std_f1)

Evaluando P_LEMMA... (esta es la más lenta, corre spaCy sobre ~133k textos por fold, puede tardar varios minutos)


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 0: Macro-F1 = 0.7786


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 1: Macro-F1 = 0.7801


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Fold 2: Macro-F1 = 0.7788
P_LEMMA -> mean=0.7791, std=0.0007
P_LEMMA run ID: d2ddee1b74b3457a86eff5f87c807bbe
🏃 View run P_LEMMA at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1/runs/d2ddee1b74b3457a86eff5f87c807bbe
🧪 View experiment at: http://ec2-100-26-91-142.compute-1.amazonaws.com:5000/#/experiments/1


'd2ddee1b74b3457a86eff5f87c807bbe'